In [1]:
import time
import random
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

START_URL = "https://www.101evler.com/kibris/satilik-daire?publish_date=last_1_week"
MAX_PAGES = 30
OUTPUT_CSV = "101evler_last1week_links.csv"

def build_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-notifications")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(35)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def set_page_url(base_url, page_num):
    parsed = urlparse(base_url)
    query = parse_qs(parsed.query)
    if page_num > 1:
        query["page"] = [str(page_num)]
        query["sort"] = ["mr"]
    new_query = urlencode(query, doseq=True)
    return urlunparse(parsed._replace(query=new_query))

def normalize_link(href):
    if not href:
        return None
    href = href.strip()
    if href.startswith("https://www.101evler.comhttps://"):
        href = href.replace("https://www.101evler.comhttps://", "https://", 1)
    elif href.startswith("https://101evler.com"):
        href = href.replace("https://101evler.com", "https://www.101evler.com", 1)
    elif href.startswith("/"):
        href = "https://www.101evler.com" + href
    return href

def is_listing_link(href):
    if not href:
        return False
    href = href.lower()
    bad_parts = [
        "javascript:",
        "/kibris/satilik-konut",
        "/kibris/kiralik-konut",
        "/firmalar",
        "/projeler",
        "/blog",
        "/rehber",
        "/iletisim",
        "/giris",
        "/kayit"
    ]
    if any(x in href for x in bad_parts):
        return False
    return "/kibris/satilik-emlak/" in href and href.endswith(".html")

def extract_listing_links(driver):
    found_links = set()
    anchors = driver.find_elements(By.TAG_NAME, "a")
    for a in anchors:
        href = normalize_link(a.get_attribute("href"))
        if is_listing_link(href):
            found_links.add(href)
    return found_links

def scrape_page(page_num):
    driver = None
    page_links = set()
    try:
        url = START_URL if page_num == 1 else set_page_url(START_URL, page_num)
        print(f"\nSayfa {page_num} aciliyor: {url}")

        driver = build_driver()
        driver.get(url)

        WebDriverWait(driver, 18).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )

        time.sleep(random.uniform(3, 5))

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.45);")
        time.sleep(random.uniform(0.8, 1.5))
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.9);")
        time.sleep(random.uniform(0.8, 1.5))
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(random.uniform(0.5, 1.0))

        page_links = extract_listing_links(driver)

        print(f"Sayfa {page_num} link sayisi: {len(page_links)}")
        if page_links:
            for link in list(sorted(page_links))[:5]:
                print(link)

    except Exception as e:
        print(f"Sayfa {page_num} hata verdi: {e}")

    finally:
        if driver:
            try:
                driver.quit()
            except:
                pass
            print(f"Sayfa {page_num} browser kapatildi")

    return page_links

def main():
    all_links = set()

    for page_num in range(1, MAX_PAGES + 1):
        links = scrape_page(page_num)
        all_links.update(links)

        if page_num < MAX_PAGES:
            sleep_between = random.uniform(1.5, 3)
            print(f"Sonraki sayfa oncesi bekleme: {sleep_between:.1f} saniye")
            time.sleep(sleep_between)

    df = pd.DataFrame({"ilan_linki": sorted(all_links)})
    df = df.dropna().drop_duplicates()
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"\nToplam benzersiz link: {len(df)}")
    print(f"Kaydedildi: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


Sayfa 1 aciliyor: https://www.101evler.com/kibris/satilik-daire?publish_date=last_1_week
Sayfa 1 link sayisi: 40
https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-518407.html
https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-518411.html
https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-518469.html
https://www.101evler.com/kibris/satilik-emlak/girne-asagi-girne-daire-428859.html
https://www.101evler.com/kibris/satilik-emlak/girne-bahceli-daire-518827.html
Sayfa 1 browser kapatildi
Sonraki sayfa oncesi bekleme: 2.1 saniye

Sayfa 2 aciliyor: https://www.101evler.com/kibris/satilik-daire?publish_date=last_1_week&page=2&sort=mr
Sayfa 2 link sayisi: 40
https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-518558.html
https://www.101evler.com/kibris/satilik-emlak/girne-catalkoy-villa-255967.html
https://www.101evler.com/kibris/satilik-emlak/girne-esentepe-daire-289984.html
https://www.101evler.com/kibris/satilik-emlak/girne-girn

In [2]:
import re
import time
import random
from datetime import datetime

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

INPUT_CSV = "101evler_last1week_links.csv"
OUTPUT_CSV = "101evler_last1week_details.csv"

def build_driver():
    options = Options()
    options.page_load_strategy = "eager"
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-notifications")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(25)
    driver.set_script_timeout(25)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def safe_text(el):
    try:
        return el.text.strip()
    except:
        return ""

def normalize_text(text):
    if text is None:
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

def normalize_label(text):
    text = normalize_text(text).lower()
    repl = {
        "ı": "i",
        "İ": "i",
        "ş": "s",
        "Ş": "s",
        "ğ": "g",
        "Ğ": "g",
        "ü": "u",
        "Ü": "u",
        "ö": "o",
        "Ö": "o",
        "ç": "c",
        "Ç": "c"
    }
    for k, v in repl.items():
        text = text.replace(k, v)
    text = text.replace(":", "").strip()
    return text

def find_first(patterns, text):
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.I)
        if m:
            return m.group(1).strip()
    return ""

def extract_ilan_id(url):
    m = re.search(r"-(\d+)\.html$", str(url))
    return m.group(1) if m else ""

def clean_price_text(text):
    if not text:
        return ""
    m = re.search(r"£\s?[\d,.]+", text)
    return m.group(0).strip() if m else ""

def extract_m2_value(text):
    if not text:
        return ""
    text_norm = normalize_text(text).lower()

    preferred_patterns = [
        r"kapalı alan[^\d]{0,20}(\d{2,4})\s?(?:m2|m²)",
        r"kapali alan[^\d]{0,20}(\d{2,4})\s?(?:m2|m²)",
        r"kullanım alanı[^\d]{0,20}(\d{2,4})\s?(?:m2|m²)",
        r"kullanim alani[^\d]{0,20}(\d{2,4})\s?(?:m2|m²)",
        r"net[^\d]{0,20}(\d{2,4})\s?(?:m2|m²)",
        r"brüt[^\d]{0,20}(\d{2,4})\s?(?:m2|m²)",
        r"brut[^\d]{0,20}(\d{2,4})\s?(?:m2|m²)",
        r"(\d{2,4})\s?(?:m2|m²)\s?kapalı alan",
        r"(\d{2,4})\s?(?:m2|m²)\s?kapali alan",
        r"(\d{2,4})\s?(?:m2|m²)\s?kullanım alanı",
        r"(\d{2,4})\s?(?:m2|m²)\s?kullanim alani",
        r"(\d{2,4})\s?(?:m2|m²)\s?net",
        r"(\d{2,4})\s?(?:m2|m²)\s?brüt",
        r"(\d{2,4})\s?(?:m2|m²)\s?brut"
    ]

    for p in preferred_patterns:
        m = re.search(p, text_norm, flags=re.I)
        if m:
            val = int(m.group(1))
            if 30 <= val <= 2000:
                return str(val)

    generic = re.findall(r"(\d{2,4})\s?(?:m2|m²)", text_norm, flags=re.I)
    nums = [int(x) for x in generic if 30 <= int(x) <= 2000]

    if not nums:
        return ""

    bad_words = ["arsa", "parsel", "donum", "dönüm", "bahce", "bahçe"]

    if any(w in text_norm for w in bad_words) and len(nums) > 1:
        small = [x for x in nums if x <= 400]
        if small:
            return str(min(small))

    return str(min(nums))

def extract_title(driver):
    selectors = ["h1", ".property-title", ".ilan-detay-baslik", ".detail-title"]
    for selector in selectors:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, selector)
            for el in els:
                txt = safe_text(el)
                if txt and len(txt) > 5:
                    return txt
        except:
            pass
    return ""

def extract_details_from_blocks(driver):
    data = {}
    try:
        blocks = driver.find_elements(By.CSS_SELECTOR, "div.text-block-141")
        for block in blocks:
            try:
                label_el = block.find_element(By.CSS_SELECTOR, ".col-5")
                value_el = block.find_element(By.CSS_SELECTOR, ".col-7")
                label = normalize_label(safe_text(label_el))
                value = normalize_text(safe_text(value_el))
                if label and value:
                    data[label] = value
            except:
                pass
    except:
        pass
    return data

def enrich_from_blocks(data, row):
    for k, v in data.items():
        k_norm = normalize_label(k)

        if not row["price"] and "fiyat" in k_norm:
            row["price"] = clean_price_text(v) or v

        if not row["m2"] and (
            "m²" in k or "m2" in k_norm or "metrekare" in k_norm or
            "kapali alan" in k_norm or "kapalı alan" in k_norm or
            "kullanim alani" in k_norm or "kullanım alanı" in k_norm or
            "net alan" in k_norm or "brut alan" in k_norm or "brüt alan" in k_norm
        ):
            m2_val = extract_m2_value(v)
            if m2_val:
                row["m2"] = m2_val

        if not row["location"] and ("konum" in k_norm or "lokasyon" in k_norm or "location" in k_norm):
            row["location"] = v

        if not row["listing_date"] and ("ilan tarihi" in k_norm or "yayin tarihi" in k_norm or "yayın tarihi" in k_norm):
            row["listing_date"] = v

        if not row["update_date"] and ("guncelleme tarihi" in k_norm or "güncelleme tarihi" in k_norm or "son guncelleme" in k_norm or "son güncelleme" in k_norm):
            row["update_date"] = v

        if not row["rooms"] and ("oda sayisi" in k_norm or "oda sayısı" in k_norm):
            row["rooms"] = v

        if not row["building_age"] and ("bina yasi" in k_norm or "bina yaşı" in k_norm):
            row["building_age"] = v

        if not row["furnished"] and ("esya durumu" in k_norm or "eşya durumu" in k_norm):
            row["furnished"] = v

    return row

def enrich_from_body(body, row):
    body_norm = normalize_text(body)

    if not row["price"]:
        row["price"] = clean_price_text(body_norm)

    if not row["m2"]:
        row["m2"] = extract_m2_value(body_norm)

    if not row["listing_date"]:
        row["listing_date"] = find_first([
            r"İlan Tarihi\s*[:\-]?\s*([0-3]?\d/[01]?\d/\d{4})",
            r"Yayın Tarihi\s*[:\-]?\s*([0-3]?\d/[01]?\d/\d{4})",
            r"Yayin Tarihi\s*[:\-]?\s*([0-3]?\d/[01]?\d/\d{4})"
        ], body_norm)

    if not row["update_date"]:
        row["update_date"] = find_first([
            r"Güncelleme Tarihi\s*[:\-]?\s*([0-3]?\d/[01]?\d/\d{4})",
            r"Guncelleme Tarihi\s*[:\-]?\s*([0-3]?\d/[01]?\d/\d{4})",
            r"Son Güncelleme\s*[:\-]?\s*([0-3]?\d/[01]?\d/\d{4})",
            r"Son Guncelleme\s*[:\-]?\s*([0-3]?\d/[01]?\d/\d{4})"
        ], body_norm)

    if not row["rooms"]:
        row["rooms"] = find_first([
            r"\b(\d+\+\d+)\b"
        ], body_norm)

    if not row["location"]:
        row["location"] = find_first([
            r"Konum\s*[:\-]?\s*([^\n]+)",
            r"Lokasyon\s*[:\-]?\s*([^\n]+)"
        ], body_norm)

    if not row["building_age"]:
        row["building_age"] = find_first([
            r"Bina Yaşı\s*[:\-]?\s*([^\n]+)",
            r"Bina Yasi\s*[:\-]?\s*([^\n]+)"
        ], body_norm)

    if not row["furnished"]:
        row["furnished"] = find_first([
            r"Eşya Durumu\s*[:\-]?\s*([^\n]+)",
            r"Esya Durumu\s*[:\-]?\s*([^\n]+)"
        ], body_norm)

    return row

def scrape_listing(url):
    driver = None

    row = {
        "ilan_id": extract_ilan_id(url),
        "ilan_linki": url,
        "title": "",
        "price": "",
        "m2": "",
        "location": "",
        "listing_date": "",
        "update_date": "",
        "rooms": "",
        "building_age": "",
        "furnished": "",
        "status": "ok",
        "scrape_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    try:
        driver = build_driver()
        driver.get(url)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )

        time.sleep(random.uniform(1.8, 3.0))

        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.45);")
            time.sleep(random.uniform(0.3, 0.8))
        except:
            pass

        row["title"] = extract_title(driver)

        data = extract_details_from_blocks(driver)
        row = enrich_from_blocks(data, row)

        body = driver.find_element(By.TAG_NAME, "body").text
        row = enrich_from_body(body, row)

    except Exception as e:
        row["status"] = f"hata: {str(e)}"
        print(f"Hata: {url} -> {e}")

    finally:
        if driver:
            try:
                driver.quit()
            except:
                pass
            print("Browser kapatildi")

    return row

def main():
    df_links = pd.read_csv(INPUT_CSV)
    urls = df_links["ilan_linki"].dropna().astype(str).tolist()

    rows = []

    for i, url in enumerate(urls, start=1):
        print(f"\n[{i}/{len(urls)}] {url}")

        row = scrape_listing(url)
        rows.append(row)

        if i % 20 == 0:
            pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
            print("Ara kayit alindi")

        time.sleep(random.uniform(0.6, 1.4))

    df_out = pd.DataFrame(rows)
    df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print("\nBitti")
    print("Kaydedildi:", OUTPUT_CSV)

if __name__ == "__main__":
    main()


[1/384] https://www.101evler.com/kibris/satilik-emlak/girne-alagadi-daire-518143.html
Browser kapatildi

[2/384] https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517558.html
Browser kapatildi

[3/384] https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517578.html
Browser kapatildi

[4/384] https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517620.html
Browser kapatildi

[5/384] https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517670.html
Browser kapatildi

[6/384] https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517705.html
Browser kapatildi

[7/384] https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517762.html
Browser kapatildi

[8/384] https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517795.html
Browser kapatildi

[9/384] https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517846.html
Browser kapatildi

[10/384] https://www.101evler.com/kibris/satil

In [3]:
import pandas as pd

df = pd.read_csv("101evler_last1week_details.csv")

print("Toplam ilan:", len(df))

print("\nEksik alan sayıları:")
print(df[["price","m2","location","listing_date","update_date","rooms","building_age","furnished"]].isna().sum())

print("\nStatus dağılımı:")
print(df["status"].value_counts(dropna=False))

print("\nÖrnek kayıtlar:")
print(df[["ilan_linki","price","m2","location","listing_date","update_date","rooms"]].head(10).to_string())

Toplam ilan: 384

Eksik alan sayıları:
price           0
m2              2
location        0
listing_date    0
update_date     0
rooms           0
building_age    1
furnished       1
dtype: int64

Status dağılımı:
status
ok    384
Name: count, dtype: int64

Örnek kayıtlar:
                                                                       ilan_linki     price     m2         location listing_date update_date rooms
0   https://www.101evler.com/kibris/satilik-emlak/girne-alagadi-daire-518143.html  £135,000  110.0   Alagadi, Girne   18/03/2026  18/03/2026   3+1
1  https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517558.html  £110,000   85.0  Alsancak, Girne   15/03/2026  20/03/2026   2+1
2  https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517578.html   £90,000   65.0  Alsancak, Girne   16/03/2026  16/03/2026   1+1
3  https://www.101evler.com/kibris/satilik-emlak/girne-alsancak-daire-517620.html   £95,000   50.0  Alsancak, Girne   16/03/2026  16/03/20

In [10]:
import pandas as pd
import numpy as np

def extract_number(val):
    if pd.isna(val):
        return np.nan
    
    s = str(val).strip().lower()
    s = s.replace(",", "")
    s = s.replace("m²", "").replace("m2", "")
    s = s.replace("£", "").replace("€", "").replace("$", "")
    s = s.replace("tl", "").replace("stg", "").replace("gbp", "")
    s = s.replace("usd", "").replace("eur", "")
    
    cleaned = ""
    for ch in s:
        if ch.isdigit() or ch == ".":
            cleaned += ch
    
    try:
        return float(cleaned) if cleaned else np.nan
    except:
        return np.nan

def clean_real_estate_data():
    input_csv = "101evler_last1week_details.csv"
    output_csv = "101evler_last1week_clean.csv"

    df = pd.read_csv(input_csv)

    print("İlk satır sayısı:", len(df))
    print("Kolonlar:", list(df.columns))

    df.columns = [c.strip().lower() for c in df.columns]

    column_map = {
        "fiyat": "fiyat",
        "price": "fiyat",
        "metrekare": "m2",
        "m2": "m2",
        "konum": "konum",
        "location": "konum",
        "ilan tarihi": "ilan_tarihi",
        "listing date": "ilan_tarihi",
        "güncelleme tarihi": "guncelleme_tarihi",
        "guncelleme tarihi": "guncelleme_tarihi",
        "update date": "guncelleme_tarihi",
        "oda sayısı": "oda_sayisi",
        "oda sayisi": "oda_sayisi",
        "room": "oda_sayisi",
        "bina yaşı": "bina_yasi",
        "bina yasi": "bina_yasi",
        "eşya durumu": "esya_durumu",
        "esya durumu": "esya_durumu",
        "ilan_linki": "ilan_linki",
        "link": "ilan_linki"
    }

    df = df.rename(columns={k: v for k, v in column_map.items() if k in df.columns})

    expected_cols = [
        "ilan_linki", "fiyat", "m2", "konum",
        "ilan_tarihi", "guncelleme_tarihi",
        "oda_sayisi", "bina_yasi", "esya_durumu"
    ]

    for col in expected_cols:
        if col not in df.columns:
            df[col] = np.nan

    df["fiyat_clean"] = df["fiyat"].apply(extract_number)
    df["m2_clean"] = df["m2"].apply(extract_number)
    df["ppm2"] = df["fiyat_clean"] / df["m2_clean"]

    df["ilan_tarihi_dt"] = pd.to_datetime(df["ilan_tarihi"], errors="coerce", dayfirst=True)
    df["guncelleme_tarihi_dt"] = pd.to_datetime(df["guncelleme_tarihi"], errors="coerce", dayfirst=True)

    for col in ["konum", "oda_sayisi", "bina_yasi", "esya_durumu", "ilan_linki"]:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"": np.nan, "nan": np.nan, "None": np.nan}).infer_objects(copy=False)

    before_dup = len(df)
    df = df.drop_duplicates(subset=["ilan_linki"], keep="first")
    print("Duplicate silinen:", before_dup - len(df))

    df = df[df["fiyat_clean"].notna()]
    df = df[df["m2_clean"].notna()]

    df = df[df["fiyat_clean"] > 10000]
    df = df[df["m2_clean"] >= 20]
    df = df[df["m2_clean"] <= 1000]

    df = df[df["ppm2"].notna()]
    df = df[df["ppm2"] > 50]
    df = df[df["ppm2"] < 10000]

    q1 = df["ppm2"].quantile(0.25)
    q3 = df["ppm2"].quantile(0.75)
    iqr = q3 - q1
    lower = max(50, q1 - 1.5 * iqr)
    upper = q3 + 1.5 * iqr

    df_clean = df[(df["ppm2"] >= lower) & (df["ppm2"] <= upper)].copy()

    df_clean["bolge"] = df_clean["konum"].astype(str).str.split("/").str[0].str.strip()
    df_clean["bolge"] = df_clean["bolge"].replace({"nan": np.nan})

    df_clean = df_clean.sort_values(by="ppm2", ascending=True)

    df_clean.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print("PPM2 alt sınır:", round(lower, 2))
    print("PPM2 üst sınır:", round(upper, 2))
    print("Final satır sayısı:", len(df_clean))
    print("Ortalama fiyat:", round(df_clean["fiyat_clean"].mean(), 2))
    print("Ortalama m2:", round(df_clean["m2_clean"].mean(), 2))
    print("Ortalama ppm2:", round(df_clean["ppm2"].mean(), 2))

    return df_clean

df_clean = clean_real_estate_data()
df_clean.head()

İlk satır sayısı: 384
Kolonlar: ['ilan_id', 'ilan_linki', 'title', 'price', 'm2', 'location', 'listing_date', 'update_date', 'rooms', 'building_age', 'furnished', 'status', 'scrape_date']
Duplicate silinen: 0
PPM2 alt sınır: 50
PPM2 üst sınır: 2761.92
Final satır sayısı: 359
Ortalama fiyat: 114940.8
Ortalama m2: 88.5
Ortalama ppm2: 1338.82


C:\Users\cagri\AppData\Local\Temp\ipykernel_31440\1644073594.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace({"": np.nan, "nan": np.nan, "None": np.nan}).infer_objects(copy=False)
C:\Users\cagri\AppData\Local\Temp\ipykernel_31440\1644073594.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace({"": np.nan, "nan": np.nan, "None": np.nan}).infer_objects(copy=False)
C:\Users\cagri\AppData\Local\Temp\ipykernel_31440\1644073594.py:80: FutureWarning: Downcasting behavior in `replac

,ilan_id,ilan_linki,title,fiyat,m2,konum,listing_date,update_date,rooms,building_age,...,guncelleme_tarihi,oda_sayisi,bina_yasi,esya_durumu,fiyat_clean,m2_clean,ppm2,ilan_tarihi_dt,guncelleme_tarihi_dt,bolge
356,518142,https://www.101evler.com/kibris/satilik-emlak/...,MAĞUSA MARAŞ KELEPİR 2+1 BODRUM KAT,"£31,500",75.0,"Maraş, Gazimağusa",18/03/2026,18/03/2026,2+1,16 - 20,...,NaN,NaN,NaN,NaN,31500.0,75.0,420.000000,NaT,NaT,"Maraş, Gazimağusa"
361,517632,https://www.101evler.com/kibris/satilik-emlak/...,SATILIK 3+1 DAİRE SOSYAL KONUTLARDA TADİLATI Y...,"£60,000",120.0,"Salamis, Gazimağusa",16/03/2026,19/03/2026,3+1,4,...,NaN,NaN,NaN,NaN,60000.0,120.0,500.000000,NaT,NaT,"Salamis, Gazimağusa"
266,518519,https://www.101evler.com/kibris/satilik-emlak/...,3+2 TÜRK KOÇANLI KELEPİR FİYATA SATILIK DAİRE,"£75,000",135.0,"Çağlayan, Lefkoşa",19/03/2026,19/03/2026,3+2,26+,...,NaN,NaN,NaN,NaN,75000.0,135.0,555.555556,NaT,NaT,"Çağlayan, Lefkoşa"
350,517930,https://www.101evler.com/kibris/satilik-emlak/...,Gazimağusa Merkez | Sakarya Bölgesinde | Eşyal...,"£65,000",110.0,"Mağusa Merkez, Gazimağusa",17/03/2026,18/03/2026,3+1,21 - 25,...,NaN,NaN,NaN,NaN,65000.0,110.0,590.909091,NaT,NaT,"Mağusa Merkez, Gazimağusa"
62,517646,https://www.101evler.com/kibris/satilik-emlak/...,ACİL SATILIK✨ GİRNE MERKEZ KASGAR BOLGESİ SATI...,"£107,000",180.0,"Girne Merkez, Girne",16/03/2026,16/03/2026,3+1,16 - 20,...,NaN,NaN,NaN,NaN,107000.0,180.0,594.444444,NaT,NaT,"Girne Merkez, Girne"


In [5]:
!pip install reportlab

In [6]:
import pandas as pd
import re

INPUT_FILE = "101evler_last1week_details.csv"
CLEAN_FILE = "101evler_last1week_clean.csv"
SUMMARY_FILE = "weekly_market_summary.csv"
DISTRICT_FILE = "weekly_district_summary.csv"
DEALS_FILE = "weekly_deals.csv"

def extract_ilan_id(url):
    match = re.search(r"-(\d+)\.html$", str(url))
    return match.group(1) if match else None

def clean_price(price):
    if pd.isna(price):
        return None
    price = str(price).replace("£", "").replace(",", "").strip()
    try:
        return float(price)
    except:
        return None

def clean_m2(val):
    if pd.isna(val):
        return None
    text = str(val).lower().replace(",", ".").strip()
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    if not match:
        return None
    try:
        num = float(match.group(1))
        if 20 <= num <= 5000:
            return num
    except:
        return None
    return None

def split_location(loc):
    if pd.isna(loc):
        return None, None
    parts = [x.strip() for x in str(loc).split(",")]
    if len(parts) >= 2:
        return parts[0], parts[-1]
    return str(loc).strip(), None

def property_type(title, url):
    text = f"{'' if pd.isna(title) else str(title).lower()} {' ' if pd.notna(url) else ''}{'' if pd.isna(url) else str(url).lower()}"
    if "penthouse" in text:
        return "penthouse"
    if "residence" in text:
        return "residence"
    if "bungalow" in text:
        return "bungalow"
    if "villa" in text:
        return "villa"
    if "müstakil" in text or "mustakil" in text:
        return "mustakil"
    if "ikiz-villa" in text or "ikiz villa" in text:
        return "villa"
    if "daire" in text:
        return "daire"
    return "unknown"

def suspicious_title(title):
    if pd.isna(title):
        return False
    t = str(title).lower()
    bad_words = [
        "arsa", "parsel", "dönüm", "donum", "metruk", "bahçe", "bahce",
        "arazi", "ticari imarlı", "ticari imarli", "imarli", "imarlı"
    ]
    return any(word in t for word in bad_words)

def suspicious_finance_title(title):
    if pd.isna(title):
        return False
    t = str(title).lower()
    finance_words = [
        "peşinat", "pesinat", "peşin", "pesin", "taksit",
        "ödeme planı", "odeme plani",
        "başlayan fiyat", "baslayan fiyat",
        "devir ücreti", "devir ucreti",
        "trafo", "vadeli",
        "aylık ödeme", "aylik odeme",
        "kalan", "geri kalan",
        "stg", "sterlin",
        "down payment", "installment", "starting from",
        "ayda", "teslimde", "anahtar teslimde",
        "50 bin", "100 bin",
        "nakit kalan", "kalan ödeme", "kalan odeme"
    ]
    return any(word in t for word in finance_words)

def valid_m2_by_type(ptype, m2):
    if pd.isna(m2):
        return False
    m2 = float(m2)
    limits = {
        "daire": (35, 220),
        "residence": (35, 180),
        "penthouse": (45, 280),
        "villa": (70, 450),
        "bungalow": (60, 350),
        "mustakil": (60, 400),
        "unknown": (30, 300)
    }
    low, high = limits.get(ptype, (30, 300))
    return low <= m2 <= high

def hard_filter(row):
    city = "" if pd.isna(row["city"]) else str(row["city"]).strip()
    ptype = row["property_type"]
    title = "" if pd.isna(row["title"]) else str(row["title"]).lower()
    price = row["price_clean"]
    m2 = row["m2_clean"]
    ppm2 = row["ppm2"]

    if pd.isna(price) or pd.isna(m2) or pd.isna(ppm2):
        return True

    if suspicious_finance_title(title):
        return True

    if ptype in ["daire", "residence"] and price < 90000:
        return True

    if ptype == "penthouse" and price < 140000:
        return True

    if city == "Lefkoşa" and ptype in ["daire", "residence", "penthouse"] and price < 120000:
        return True

    if city == "Girne" and ptype in ["daire", "residence", "penthouse"] and price < 95000:
        return True

    if city == "İskele" and ptype in ["daire", "residence", "penthouse"] and price < 70000:
        return True

    if city == "Gazimağusa" and ptype in ["daire", "residence", "penthouse"] and price < 70000:
        return True

    if ptype in ["daire", "residence", "penthouse"] and ppm2 < 900:
        return True

    if price < 60000 and m2 > 60:
        return True

    if price < 80000 and m2 > 100:
        return True

    if ptype in ["villa", "bungalow", "mustakil"] and m2 >= 300 and ppm2 < 800:
        return True

    return False

df = pd.read_csv(INPUT_FILE)

df["ilan_id"] = df["ilan_linki"].apply(extract_ilan_id)
df["price_clean"] = df["price"].apply(clean_price)
df["m2_clean"] = df["m2"].apply(clean_m2)
df[["district", "city"]] = df["location"].apply(lambda x: pd.Series(split_location(x)))

df = df.drop_duplicates(subset="ilan_id")
df = df[df["price_clean"].notna()]
df = df[df["m2_clean"].notna()]
df = df[df["m2_clean"] > 0]

df["property_type"] = df.apply(lambda row: property_type(row["title"], row["ilan_linki"]), axis=1)
df["ppm2"] = df["price_clean"] / df["m2_clean"]

df = df[(df["price_clean"] >= 10000) & (df["price_clean"] <= 10000000)]
df = df[(df["m2_clean"] >= 20) & (df["m2_clean"] <= 5000)]
df = df[(df["ppm2"] >= 100) & (df["ppm2"] <= 20000)]

df = df[~df["title"].apply(suspicious_title)].copy()
df = df[~df["title"].apply(suspicious_finance_title)].copy()

df = df[
    df.apply(lambda row: valid_m2_by_type(row["property_type"], row["m2_clean"]), axis=1)
].copy()

df = df[
    ~(
        (df["property_type"].isin(["daire", "residence"])) &
        (df["m2_clean"] > 200)
    )
].copy()

df = df[
    ~(
        (df["property_type"].isin(["daire", "residence", "penthouse"])) &
        (df["ppm2"] < 900)
    )
].copy()

df = df[~df.apply(hard_filter, axis=1)].copy()

df.to_csv(CLEAN_FILE, index=False, encoding="utf-8-sig")

summary = df.groupby("city", dropna=False).agg(
    ilan_sayisi=("ilan_id", "count"),
    ortalama_fiyat=("price_clean", "mean"),
    medyan_fiyat=("price_clean", "median"),
    ortalama_m2=("m2_clean", "mean"),
    ortalama_ppm2=("ppm2", "mean"),
    medyan_ppm2=("ppm2", "median"),
    min_ppm2=("ppm2", "min"),
    max_ppm2=("ppm2", "max")
).reset_index()

summary = summary.sort_values("ortalama_ppm2", ascending=False)
summary.to_csv(SUMMARY_FILE, index=False, encoding="utf-8-sig")

district_summary = df.groupby(["city", "district"], dropna=False).agg(
    ilan_sayisi=("ilan_id", "count"),
    ortalama_fiyat=("price_clean", "mean"),
    medyan_fiyat=("price_clean", "median"),
    ortalama_m2=("m2_clean", "mean"),
    ortalama_ppm2=("ppm2", "mean"),
    medyan_ppm2=("ppm2", "median")
).reset_index()

district_summary = district_summary.sort_values(["city", "ortalama_ppm2"], ascending=[True, False])
district_summary.to_csv(DISTRICT_FILE, index=False, encoding="utf-8-sig")

district_avg_map = district_summary.set_index("district")["ortalama_ppm2"].to_dict()

df["district_avg_ppm2"] = df["district"].map(district_avg_map)
df["deal_score"] = df["ppm2"] / df["district_avg_ppm2"]
df["discount_vs_district_pct"] = (1 - df["deal_score"]) * 100

deals = df[df["deal_score"] < 0.85].copy()
deals = deals.sort_values(["discount_vs_district_pct", "ppm2"], ascending=[False, True])

deal_cols = [
    "ilan_id",
    "ilan_linki",
    "title",
    "city",
    "district",
    "price_clean",
    "m2_clean",
    "ppm2",
    "district_avg_ppm2",
    "deal_score",
    "discount_vs_district_pct",
    "property_type",
    "location",
    "listing_date",
    "update_date",
    "rooms",
    "building_age",
    "furnished"
]

deals[deal_cols].to_csv(DEALS_FILE, index=False, encoding="utf-8-sig")

print("Clean veri:", CLEAN_FILE)
print("City summary:", SUMMARY_FILE)
print("District summary:", DISTRICT_FILE)
print("Deals:", DEALS_FILE)
print()
print(df[["price_clean", "m2_clean", "ppm2"]].describe())
print()
print(df[["price_clean", "m2_clean", "ppm2"]].isna().sum())
print()
print(df.sort_values("price_clean")[["ilan_linki", "title", "city", "district", "property_type", "price_clean", "m2_clean", "ppm2"]].head(20).to_string())

Clean veri: 101evler_last1week_clean.csv
City summary: weekly_market_summary.csv
District summary: weekly_district_summary.csv
Deals: weekly_deals.csv

         price_clean    m2_clean         ppm2
count     153.000000  153.000000   153.000000
mean   152483.483660   88.816993  1808.737803
std     51649.941496   23.066391   750.515651
min     90000.000000   42.000000   925.925926
25%    115000.000000   76.000000  1312.500000
50%    139000.000000   87.000000  1628.571429
75%    179900.000000  100.000000  2133.333333
max    345000.000000  150.000000  6960.555556

price_clean    0
m2_clean       0
ppm2           0
dtype: int64

                                                                               ilan_linki                                                                                         title        city       district property_type  price_clean  m2_clean         ppm2
185     https://www.101evler.com/kibris/satilik-emlak/iskele-long-beach-daire-518358.html                  

In [9]:
import os
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import cm
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics

CLEAN_FILE = "101evler_last1week_clean.csv"
PDF_FILE = "weekly_real_estate_report.pdf"

CHART_CITY_PPM2 = "chart_city_ppm2.png"
CHART_CITY_COUNT = "chart_city_count.png"
CHART_TOP_DISTRICTS = "chart_top_districts.png"
CHART_CHEAP_DISTRICTS = "chart_cheap_districts.png"

MIN_DISTRICT_COUNT = 5
TOP_N_CITIES = 10
TOP_N_DISTRICTS = 10
TOP_N_DEALS = 18

clean_df = pd.read_csv(CLEAN_FILE)
clean_df.columns = [str(c).strip().lower() for c in clean_df.columns]

def pick_font_paths():
    candidates = [
        ("C:/Windows/Fonts/arial.ttf", "C:/Windows/Fonts/arialbd.ttf"),
        ("C:/Windows/Fonts/calibri.ttf", "C:/Windows/Fonts/calibrib.ttf"),
        ("C:/Windows/Fonts/tahoma.ttf", "C:/Windows/Fonts/tahomabd.ttf"),
        ("DejaVuSans.ttf", "DejaVuSans-Bold.ttf"),
        ("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf")
    ]
    for regular, bold in candidates:
        if os.path.exists(regular) and os.path.exists(bold):
            return regular, bold
    raise FileNotFoundError("Unicode font bulunamadı")

regular_font_path, bold_font_path = pick_font_paths()
pdfmetrics.registerFont(TTFont("TR-Regular", regular_font_path))
pdfmetrics.registerFont(TTFont("TR-Bold", bold_font_path))

def find_col(df, candidates, required=True):
    cols = list(df.columns)
    for cand in candidates:
        if cand in cols:
            return cand
    for cand in candidates:
        for col in cols:
            if cand in col:
                return col
    if required:
        raise KeyError(f"Kolon bulunamadı. Arananlar: {candidates} | Mevcut kolonlar: {cols}")
    return None

def extract_number(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    s = s.replace(",", "")
    s = s.replace("m²", "").replace("m2", "")
    s = s.replace("£", "").replace("€", "").replace("$", "")
    s = s.replace("tl", "").replace("stg", "").replace("gbp", "")
    s = s.replace("usd", "").replace("eur", "")
    cleaned = ""
    for ch in s:
        if ch.isdigit() or ch == ".":
            cleaned += ch
    try:
        return float(cleaned) if cleaned else np.nan
    except:
        return np.nan

def fmt_num(x):
    if pd.isna(x):
        return "-"
    return f"{x:,.0f}"

def fmt_pct(x):
    if pd.isna(x):
        return "-"
    return f"{x:.1f}%"

def clean_text(x):
    if pd.isna(x):
        return "-"
    return str(x).replace("\xa0", " ").strip()

def clean_district_name(x):
    if pd.isna(x):
        return "-"
    x = str(x).split(",")[0].strip()
    x = " ".join(x.split())
    return " ".join([w.capitalize() for w in x.lower().split()])

def short_link(url):
    url = clean_text(url)
    url = re.sub(r"^https?://(www\.)?", "", url)
    if len(url) > 70:
        return url[:67] + "..."
    return url

def wrap_text(text, max_len=82):
    text = clean_text(text)
    words = text.split()
    lines = []
    current = ""
    for w in words:
        test = f"{current} {w}".strip()
        if len(test) <= max_len:
            current = test
        else:
            if current:
                lines.append(current)
            current = w
    if current:
        lines.append(current)
    return lines

def wrap_label(text, width=16):
    return "\n".join(textwrap.wrap(str(text), width=width))

def draw_page_title(c, title):
    c.setFont("TR-Bold", 18)
    c.drawString(2 * cm, 27.5 * cm, clean_text(title))

def draw_line(c, y):
    c.line(2 * cm, y, 19 * cm, y)

def draw_text(c, x, y, text, size=10, bold=False):
    c.setFont("TR-Bold" if bold else "TR-Regular", size)
    c.drawString(x, y, clean_text(text))

def draw_clickable_link(c, x, y, url, label, size=8):
    label = clean_text(label)
    url = clean_text(url)
    c.setFont("TR-Regular", size)
    c.drawString(x, y, label)
    text_width = pdfmetrics.stringWidth(label, "TR-Regular", size)
    c.linkURL(url, (x, y - 2, x + text_width, y + size + 2), relative=0)

def new_page(c, title):
    c.showPage()
    draw_page_title(c, title)
    draw_line(c, 27.1 * cm)
    return 26.2 * cm

konum_col = find_col(clean_df, ["konum", "location", "bolge_detay", "district", "adres"])
link_col = find_col(clean_df, ["ilan_linki", "link", "url", "listing_url", "ilan url"])

fiyat_clean_col = find_col(clean_df, ["fiyat_clean"], required=False)
if not fiyat_clean_col:
    fiyat_raw_col = find_col(clean_df, ["fiyat", "price"])
    clean_df["fiyat_clean"] = clean_df[fiyat_raw_col].apply(extract_number)
    fiyat_clean_col = "fiyat_clean"

m2_clean_col = find_col(clean_df, ["m2_clean"], required=False)
if not m2_clean_col:
    m2_raw_col = find_col(clean_df, ["m2", "metrekare", "area"])
    clean_df["m2_clean"] = clean_df[m2_raw_col].apply(extract_number)
    m2_clean_col = "m2_clean"

ppm2_col = find_col(clean_df, ["ppm2"], required=False)
if not ppm2_col:
    clean_df["ppm2"] = clean_df[fiyat_clean_col] / clean_df[m2_clean_col]
    ppm2_col = "ppm2"

clean_df["konum"] = clean_df[konum_col].astype(str).str.strip()
clean_df["ilan_linki"] = clean_df[link_col].astype(str).str.strip()
clean_df["fiyat_clean"] = pd.to_numeric(clean_df[fiyat_clean_col], errors="coerce")
clean_df["m2_clean"] = pd.to_numeric(clean_df[m2_clean_col], errors="coerce")
clean_df["ppm2"] = pd.to_numeric(clean_df[ppm2_col], errors="coerce")
clean_df["bolge"] = clean_df["konum"].apply(clean_district_name)

clean_df = clean_df[
    clean_df["konum"].notna() &
    clean_df["ilan_linki"].notna() &
    clean_df["fiyat_clean"].notna() &
    clean_df["m2_clean"].notna() &
    clean_df["ppm2"].notna()
].copy()

district_summary = (
    clean_df.groupby(["bolge", "konum"])
    .agg(
        ilan_sayisi=("ilan_linki", "count"),
        ortalama_fiyat=("fiyat_clean", "mean"),
        medyan_fiyat=("fiyat_clean", "median"),
        ortalama_m2=("m2_clean", "mean"),
        ortalama_ppm2=("ppm2", "mean")
    )
    .reset_index()
)

district_summary = district_summary[district_summary["ilan_sayisi"] >= MIN_DISTRICT_COUNT].copy()

city_summary = (
    district_summary.groupby("bolge")
    .agg(
        ilan_sayisi=("ilan_sayisi", "sum"),
        ortalama_fiyat=("ortalama_fiyat", "mean"),
        medyan_fiyat=("medyan_fiyat", "mean"),
        ortalama_m2=("ortalama_m2", "mean"),
        ortalama_ppm2=("ortalama_ppm2", "mean")
    )
    .reset_index()
    .sort_values("ortalama_ppm2", ascending=False)
)

district_avg_map = district_summary.set_index("konum")["ortalama_ppm2"].to_dict()

deals_df = clean_df.copy()
deals_df["district_avg_ppm2"] = deals_df["konum"].map(district_avg_map)
deals_df["discount_vs_district_pct"] = (
    (deals_df["district_avg_ppm2"] - deals_df["ppm2"]) / deals_df["district_avg_ppm2"]
) * 100

text_cols = []
for col in ["property_type", "emlak_tipi", "kategori", "listing_type", "tur", "tip"]:
    if col in deals_df.columns:
        text_cols.append(deals_df[col].astype(str).str.lower())

combined_text = deals_df["konum"].astype(str).str.lower() + " " + deals_df["ilan_linki"].astype(str).str.lower()
for s in text_cols:
    combined_text = combined_text + " " + s

arsa_mask = combined_text.str.contains(r"arsa|tarla|land|plot", na=False)
deals_df = deals_df[~arsa_mask].copy()

deals_df = deals_df[
    (deals_df["district_avg_ppm2"].notna()) &
    (deals_df["discount_vs_district_pct"] > 8) &
    (deals_df["m2_clean"] >= 45) &
    (deals_df["fiyat_clean"] >= 30000)
].copy()

if "ilan_id" in deals_df.columns:
    deals_df["ilan_id"] = deals_df["ilan_id"].astype(str).str.strip()
    deals_df = deals_df.drop_duplicates(subset=["ilan_id"], keep="first")
else:
    deals_df = deals_df.drop_duplicates(subset=["ilan_linki"], keep="first")

deals_df = deals_df.drop_duplicates(subset=["konum", "fiyat_clean", "m2_clean"], keep="first")
deals_df = deals_df.sort_values(["discount_vs_district_pct", "ppm2"], ascending=[False, True]).head(TOP_N_DEALS)

top_city_ppm2 = city_summary.sort_values("ortalama_ppm2", ascending=False).head(TOP_N_CITIES).copy()
top_city_ppm2["label"] = top_city_ppm2["bolge"].apply(lambda x: wrap_label(x, 14))

plt.figure(figsize=(10, 6))
plt.barh(top_city_ppm2["label"], top_city_ppm2["ortalama_ppm2"])
plt.title(f"En Yüksek Ortalama m² Fiyatlı {TOP_N_CITIES} Bölge")
plt.xlabel("£ / m²")
plt.ylabel("Bölge")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(CHART_CITY_PPM2, dpi=220, bbox_inches="tight")
plt.close()

top_city_count = city_summary.sort_values("ilan_sayisi", ascending=False).head(TOP_N_CITIES).copy()
top_city_count["label"] = top_city_count["bolge"].apply(lambda x: wrap_label(x, 14))

plt.figure(figsize=(10, 6))
plt.barh(top_city_count["label"], top_city_count["ilan_sayisi"])
plt.title(f"En Fazla İlan Olan {TOP_N_CITIES} Bölge")
plt.xlabel("İlan Sayısı")
plt.ylabel("Bölge")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(CHART_CITY_COUNT, dpi=220, bbox_inches="tight")
plt.close()

top_expensive = district_summary.sort_values("ortalama_ppm2", ascending=False).head(TOP_N_DISTRICTS).copy()
top_expensive["label"] = top_expensive["konum"].apply(lambda x: wrap_label(clean_district_name(x), 20))

plt.figure(figsize=(10, 6))
plt.barh(top_expensive["label"], top_expensive["ortalama_ppm2"])
plt.title(f"En Pahalı {TOP_N_DISTRICTS} Bölge")
plt.xlabel("£ / m²")
plt.ylabel("Bölge")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(CHART_TOP_DISTRICTS, dpi=220, bbox_inches="tight")
plt.close()

top_cheap = district_summary.sort_values("ortalama_ppm2", ascending=True).head(TOP_N_DISTRICTS).copy()
top_cheap["label"] = top_cheap["konum"].apply(lambda x: wrap_label(clean_district_name(x), 20))

plt.figure(figsize=(10, 6))
plt.barh(top_cheap["label"], top_cheap["ortalama_ppm2"])
plt.title(f"En Ucuz {TOP_N_DISTRICTS} Bölge")
plt.xlabel("£ / m²")
plt.ylabel("Bölge")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(CHART_CHEAP_DISTRICTS, dpi=220, bbox_inches="tight")
plt.close()

c = canvas.Canvas(PDF_FILE, pagesize=A4)

draw_page_title(c, "Kuzey Kıbrıs Haftalık Emlak Raporu")
draw_line(c, 27.1 * cm)

total_listings = len(clean_df)
avg_price = clean_df["fiyat_clean"].mean()
median_price = clean_df["fiyat_clean"].median()
avg_m2 = clean_df["m2_clean"].mean()
median_m2 = clean_df["m2_clean"].median()
avg_ppm2 = clean_df["ppm2"].mean()
median_ppm2 = clean_df["ppm2"].median()

top_city = city_summary.sort_values("ortalama_ppm2", ascending=False).head(1)
cheap_city = city_summary.sort_values("ortalama_ppm2", ascending=True).head(1)

y = 25.8 * cm
draw_text(c, 2 * cm, y, "Genel Özet", 13, True)
y -= 0.85 * cm

summary_lines = [
    f"Toplam temiz ilan sayısı: {fmt_num(total_listings)}",
    f"Ortalama fiyat: £{fmt_num(avg_price)}",
    f"Medyan fiyat: £{fmt_num(median_price)}",
    f"Ortalama büyüklük: {fmt_num(avg_m2)} m²",
    f"Medyan büyüklük: {fmt_num(median_m2)} m²",
    f"Ortalama m² fiyatı: £{fmt_num(avg_ppm2)}",
    f"Medyan m² fiyatı: £{fmt_num(median_ppm2)}",
    f"Fırsat ilanı sayısı: {fmt_num(len(deals_df))}",
    f"Piyasa özeti ve grafiklerde minimum ilan eşiği: {MIN_DISTRICT_COUNT}"
]

if len(top_city) > 0:
    summary_lines.append(f"En pahalı bölge: {clean_text(top_city.iloc[0]['bolge'])} (£{fmt_num(top_city.iloc[0]['ortalama_ppm2'])}/m²)")

if len(cheap_city) > 0:
    summary_lines.append(f"En ucuz bölge: {clean_text(cheap_city.iloc[0]['bolge'])} (£{fmt_num(cheap_city.iloc[0]['ortalama_ppm2'])}/m²)")

for line in summary_lines:
    draw_text(c, 2 * cm, y, line, 11)
    y -= 0.62 * cm

y -= 0.3 * cm
draw_text(c, 2 * cm, y, "Bölge Bazlı Piyasa Özeti", 13, True)
y -= 0.8 * cm

draw_text(c, 2.0 * cm, y, "Bölge", 9, True)
draw_text(c, 5.8 * cm, y, "İlan", 9, True)
draw_text(c, 7.1 * cm, y, "Ort. Fiyat", 9, True)
draw_text(c, 10.0 * cm, y, "Medyan", 9, True)
draw_text(c, 12.6 * cm, y, "Ort. m²", 9, True)
draw_text(c, 15.0 * cm, y, "Ort. m² Fiyatı", 9, True)
y -= 0.45 * cm
draw_line(c, y)
y -= 0.45 * cm

for _, row in city_summary.iterrows():
    if y < 3 * cm:
        y = new_page(c, "Bölge Bazlı Piyasa Özeti")
        draw_text(c, 2.0 * cm, y, "Bölge", 9, True)
        draw_text(c, 5.8 * cm, y, "İlan", 9, True)
        draw_text(c, 7.1 * cm, y, "Ort. Fiyat", 9, True)
        draw_text(c, 10.0 * cm, y, "Medyan", 9, True)
        draw_text(c, 12.6 * cm, y, "Ort. m²", 9, True)
        draw_text(c, 15.0 * cm, y, "Ort. m² Fiyatı", 9, True)
        y -= 0.45 * cm
        draw_line(c, y)
        y -= 0.45 * cm

    draw_text(c, 2.0 * cm, y, clean_district_name(row["bolge"]), 8)
    draw_text(c, 5.8 * cm, y, fmt_num(row["ilan_sayisi"]), 8)
    draw_text(c, 7.1 * cm, y, f"£{fmt_num(row['ortalama_fiyat'])}", 8)
    draw_text(c, 10.0 * cm, y, f"£{fmt_num(row['medyan_fiyat'])}", 8)
    draw_text(c, 12.6 * cm, y, fmt_num(row["ortalama_m2"]), 8)
    draw_text(c, 15.0 * cm, y, f"£{fmt_num(row['ortalama_ppm2'])}", 8)
    y -= 0.52 * cm

y = new_page(c, "Grafikler")
c.drawImage(CHART_CITY_PPM2, 2 * cm, 15.1 * cm, width=16.5 * cm, height=7.7 * cm, preserveAspectRatio=True, mask="auto")
c.drawImage(CHART_CITY_COUNT, 2 * cm, 5.3 * cm, width=16.5 * cm, height=7.7 * cm, preserveAspectRatio=True, mask="auto")

y = new_page(c, "Bölge Grafikleri")
c.drawImage(CHART_TOP_DISTRICTS, 2 * cm, 15.1 * cm, width=16.5 * cm, height=7.7 * cm, preserveAspectRatio=True, mask="auto")
c.drawImage(CHART_CHEAP_DISTRICTS, 2 * cm, 5.3 * cm, width=16.5 * cm, height=7.7 * cm, preserveAspectRatio=True, mask="auto")

y = new_page(c, "Fırsat İlanları")
draw_text(c, 2 * cm, y, f"En Güçlü {TOP_N_DEALS} Fırsat İlanı", 12, True)
y -= 0.9 * cm

for _, row in deals_df.iterrows():
    line1 = clean_district_name(row["konum"])
    line2 = f"Fiyat: £{fmt_num(row['fiyat_clean'])} | Alan: {fmt_num(row['m2_clean'])} m² | m²: £{fmt_num(row['ppm2'])}"
    line3 = f"Bölge ort.: £{fmt_num(row['district_avg_ppm2'])}/m² | İskonto: {fmt_pct(row['discount_vs_district_pct'])}"
    link = clean_text(row["ilan_linki"])

    lines1 = wrap_text(line1, max_len=68)
    lines2 = wrap_text(line2, max_len=76)
    lines3 = wrap_text(line3, max_len=76)

    needed_lines = len(lines1) + len(lines2) + len(lines3) + 2
    needed_height = needed_lines * 0.44 * cm + 0.45 * cm

    if y - needed_height < 2.7 * cm:
        y = new_page(c, "Fırsat İlanları")

    for txt in lines1:
        draw_text(c, 2.2 * cm, y, txt, 9, True)
        y -= 0.44 * cm

    for txt in lines2:
        draw_text(c, 2.2 * cm, y, txt, 9)
        y -= 0.44 * cm

    for txt in lines3:
        draw_text(c, 2.2 * cm, y, txt, 9)
        y -= 0.44 * cm

    draw_clickable_link(c, 2.2 * cm, y, link, short_link(link), size=8)
    y -= 0.6 * cm
    draw_line(c, y)
    y -= 0.45 * cm

c.save()

print("PDF created:", PDF_FILE)
print("Kullanılan konum kolonu:", konum_col)
print("Kullanılan link kolonu:", link_col)
print("Minimum ilan eşiği:", MIN_DISTRICT_COUNT)
print("Fırsat ilanı sayısı:", len(deals_df))

PDF created: weekly_real_estate_report.pdf
Kullanılan konum kolonu: konum
Kullanılan link kolonu: ilan_linki
Minimum ilan eşiği: 5
Fırsat ilanı sayısı: 18
